# SpecMAE Colab Runner

This notebook is designed to run SpecMAE training on a free Colab GPU.

It will:
- verify GPU runtime
- mount Google Drive (optional, recommended)
- clone/update the repo
- install dependencies
- run a YAML-config training preset
- save artifacts (checkpoints, plots, metrics, examples)

In [ ]:
import json
import os
import pathlib
import shlex
import subprocess
import sys

def run(cmd: str, cwd: str | None = None) -> None:
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

print(sys.version)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. In Colab: Runtime -> Change runtime type -> GPU.')

In [ ]:
# ===== User settings =====
REPO_URL = 'https://github.com/jmbrown89/SpecMAE.git'
REPO_BRANCH = 'main'
USE_DRIVE = True
DRIVE_BASE = '/content/drive/MyDrive/specmae'
LOCAL_BASE = '/content'
CONFIG_PATH = 'configs/medmnist_long_stable_linear.yaml'  # change preset here
RUN_NAME = 'colab_long_stable_linear'
EPOCHS_OVERRIDE = None  # e.g. 30 for full run, 1 for quick smoke
LIMIT_SAMPLES_OVERRIDE = None  # e.g. 256 or None to use YAML

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = pathlib.Path(DRIVE_BASE)
else:
    BASE_DIR = pathlib.Path(LOCAL_BASE) / 'specmae_workspace'

BASE_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = BASE_DIR / 'SpecMAE'
ARTIFACTS_ROOT = BASE_DIR / 'runs'
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
print('BASE_DIR:', BASE_DIR)
print('REPO_DIR:', REPO_DIR)
print('ARTIFACTS_ROOT:', ARTIFACTS_ROOT)

In [ ]:
if REPO_DIR.exists():
    run('git fetch --all', cwd=str(REPO_DIR))
    run(f'git checkout {shlex.quote(REPO_BRANCH)}', cwd=str(REPO_DIR))
    run('git pull --ff-only', cwd=str(REPO_DIR))
else:
    run(f'git clone --branch {shlex.quote(REPO_BRANCH)} {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}')

print('Repo ready at', REPO_DIR)

In [ ]:
run('python -m pip install --upgrade pip', cwd=str(REPO_DIR))
run('python -m pip install -r requirements.txt', cwd=str(REPO_DIR))
run('python -m pip install pyyaml', cwd=str(REPO_DIR))

In [ ]:
run('python -m pytest -q', cwd=str(REPO_DIR))

In [ ]:
train_cmd = [
    'python', '-m', 'specmae.training.train',
    '--config', CONFIG_PATH,
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--artifacts-root', str(ARTIFACTS_ROOT),
    '--run-name', RUN_NAME,
]

if EPOCHS_OVERRIDE is not None:
    train_cmd += ['--epochs', str(EPOCHS_OVERRIDE)]
if LIMIT_SAMPLES_OVERRIDE is not None:
    train_cmd += ['--limit-samples', str(LIMIT_SAMPLES_OVERRIDE)]

run(' '.join(shlex.quote(x) for x in train_cmd), cwd=str(REPO_DIR))

In [ ]:
run_dir = ARTIFACTS_ROOT / RUN_NAME
summary_path = run_dir / 'metrics' / 'summary.json'
report_path = run_dir / 'report.md'

print('Run dir:', run_dir)
print('Exists:', run_dir.exists())

if summary_path.exists():
    with summary_path.open('r', encoding='utf-8') as f:
        summary = json.load(f)
    print('\nSummary JSON:')
    print(json.dumps(summary, indent=2))
else:
    print('No summary.json found yet.')

if report_path.exists():
    print('\nReport preview:')
    print(report_path.read_text(encoding='utf-8')[:2000])

In [ ]:
# Optional: create a zip bundle for easy download/sharing
zip_path = ARTIFACTS_ROOT / f'{RUN_NAME}.zip'
run(f"cd {shlex.quote(str(ARTIFACTS_ROOT))} ; zip -r {shlex.quote(zip_path.name)} {shlex.quote(RUN_NAME)}")
print('Created:', zip_path)

## Alternate Presets

Set `CONFIG_PATH` in Cell 4 to one of:
- `configs/medmnist_long_stable_linear.yaml`
- `configs/medmnist_long_moderate_epoch.yaml`
- `configs/medmnist_long_aggressive_step.yaml`
- `configs/medmnist_long_fixed_baseline.yaml`